In [3]:
import cv2
import os
import io
from PIL import Image
import numpy as np
import pillow_heif
from pillow_heif import register_heif_opener
from pillow_heif import HeifImage

register_heif_opener()

img_bgr = cv2.imread("frame.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img = Image.fromarray(img_rgb)

# Save as .heic with correct format
img.save("output.heic", format="HEIF", quality=95)


# === CONFIGURATION ===
video_path = "IMG_2820.MOV"  # Your Live Photo video
output_folder = "top5_sharp_frames"
frame_interval = 1  # Set higher (e.g., 3) for faster scan

os.makedirs(output_folder, exist_ok=True)

In [5]:
# === SHARPNESS FUNCTION ===
def calculate_sharpness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()

# === PROCESS VIDEO ===
cap = cv2.VideoCapture(video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total frames: {frame_count}")

sharpness_list = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        sharpness = calculate_sharpness(frame)
        sharpness_list.append((frame_idx, sharpness, frame.copy()))

    frame_idx += 1

cap.release()

# === SORT TOP 5 FRAMES ===
top_frames = sorted(sharpness_list, key=lambda x: x[1], reverse=True)[:5]

# === EXPORT & SHOW ===
for i, (idx, sharpness, frame) in enumerate(top_frames):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    output_path = os.path.join(output_folder, f"frame_{idx}_sharpness_{int(sharpness)}.jpg")
    Image.fromarray(rgb_frame).save(output_path)
    print(f"[{i+1}] Saved: {output_path} (Sharpness: {sharpness:.2f})")

print("✔ Top 5 sharpest frames exported.")


Total frames: 58
[1] Saved: top5_sharp_frames/frame_55_sharpness_1110.jpg (Sharpness: 1110.24)
[2] Saved: top5_sharp_frames/frame_57_sharpness_1030.jpg (Sharpness: 1030.96)
[3] Saved: top5_sharp_frames/frame_37_sharpness_1004.jpg (Sharpness: 1004.66)
[4] Saved: top5_sharp_frames/frame_34_sharpness_982.jpg (Sharpness: 982.17)
[5] Saved: top5_sharp_frames/frame_51_sharpness_978.jpg (Sharpness: 978.63)
✔ Top 5 sharpest frames exported.


In [5]:
# === SHARPNESS FUNCTION ===
def calculate_sharpness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()

# === PROCESS VIDEO ===
cap = cv2.VideoCapture(video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total frames: {frame_count}")

sharpness_list = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        sharpness = calculate_sharpness(frame)
        sharpness_list.append((frame_idx, sharpness, frame.copy()))

    frame_idx += 1

cap.release()

# === SELECT TOP N SHARPEST FRAMES (initial filter) ===
initial_top_frames = sorted(sharpness_list, key=lambda x: x[1], reverse=True)[:20]


# === EXPORT AND RANK BY FILE SIZE ===
saved_frames_info = []

for idx, sharpness, frame in sharpness_list:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)

    # Convert PIL Image to HEIF
    heif_img = HeifImage.from_pillow(img)
    heic_path = os.path.join(output_folder, f"frame_{idx}_sharpness_{int(sharpness)}.heic")

    # Save using pillow_heif directly
    heif_img.save(heic_path, quality=95)
    file_size = os.path.getsize(heic_path)

    saved_frames_info.append((heic_path, idx, sharpness, file_size))


# === SORT BY FILE SIZE ===
top5_by_size = sorted(saved_frames_info, key=lambda x: x[3], reverse=True)[:5]

# === FINAL REPORT ===
print("📦 Top 5 frames by file size:")
for i, (path, idx, sharpness, size) in enumerate(top5_by_size, 1):
    print(f"[{i}] {os.path.basename(path)} | Sharpness: {sharpness:.2f} | Size: {size/1024:.1f} KB")

print("✔ Done. Top 5 frames exported in HEIC format.")


Total frames: 58


AttributeError: type object 'HeifImage' has no attribute 'from_pillow'

In [ ]:
import cv2
import os
from PIL import Image
from pillow_heif import register_heif_opener
import numpy as np

# === REGISTER HEIF FORMAT SUPPORT ===
register_heif_opener()

# === CONFIGURATION ===
video_path = "IMG_2820.MOV"  # Path to your Live Photo video
output_folder = "top5_sharp_frames_heic"
frame_interval = 1  # Higher = faster processing (e.g. 3 for every 3rd frame)

os.makedirs(output_folder, exist_ok=True)

# === FUNCTION TO MEASURE SHARPNESS ===
def calculate_sharpness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()

# === READ VIDEO AND EXTRACT FRAMES ===
cap = cv2.VideoCapture(video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"🎞️ Processing video: {video_path}")
print(f"🔢 Total frames: {frame_count}, FPS: {fps:.2f}")

sharpness_list = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        sharpness = calculate_sharpness(frame)
        sharpness_list.append((frame_idx, sharpness, frame.copy()))

    frame_idx += 1

cap.release()

# === EXPORT FRAMES AS .HEIC AND RECORD FILE SIZE ===
saved_frames_info = []

print(f"💾 Exporting {len(sharpness_list)} frames to HEIC format...")

for idx, sharpness, frame in sharpness_list:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)

    heic_filename = f"frame_{idx}_sharpness_{int(sharpness)}.heic"
    heic_path = os.path.join(output_folder, heic_filename)

    try:
        img.save(heic_path, format="HEIF", quality=95)
        file_size = os.path.getsize(heic_path)
        saved_frames_info.append((heic_path, idx, sharpness, file_size))
    except Exception as e:
        print(f"❌ Failed to save frame {idx}: {e}")

# === RANK TOP 5 FRAMES BY FILE SIZE ===
top_by_size = sorted(saved_frames_info, key=lambda x: x[3], reverse=True)[:5]

print("\n📦 Top 5 HEIC frames (by file size):")
for i, (path, idx, sharpness, size) in enumerate(top_by_size):
    print(f"[{i+1}] {os.path.basename(path)} - Sharpness: {sharpness:.2f}, Size: {size / 1024:.1f} KB")

print(f"\n✔ Done. All HEIC frames saved to: {output_folder}")


🎞️ Processing video: IMG_2820.MOV
🔢 Total frames: 58, FPS: 21.38


In [1]:
import cv2
import os
from PIL import Image
import numpy as np

# === CONFIGURATION ===
video_path = "IMG_2820.MOV"  # Replace with your Live Photo video path
output_folder = "all_frames_png"
frame_interval = 1           # Extract every N frames (1 = all frames)

os.makedirs(output_folder, exist_ok=True)

# === OPEN VIDEO ===
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {video_path}")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"🎞️ Video: {video_path}")
print(f"📏 Resolution: {width}x{height}")
print(f"🔢 Total frames: {frame_count}, FPS: {fps:.2f}")
print(f"💾 Exporting frames to: {output_folder}")

# === EXTRACT AND SAVE FRAMES ===
frame_idx = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        # Convert BGR (OpenCV) to RGB (PIL)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)

        filename = os.path.join(output_folder, f"frame_{frame_idx:04d}.png")
        img.save(filename, format="PNG")
        saved += 1

    frame_idx += 1

cap.release()
print(f"✅ Done. {saved} frames saved as PNG.")


🎞️ Video: IMG_2820.MOV
📏 Resolution: 1920x1440
🔢 Total frames: 58, FPS: 21.38
💾 Exporting frames to: all_frames_png
✅ Done. 58 frames saved as PNG.


In [3]:
from PIL import Image
from pillow_heif import register_heif_opener

# Register HEIF/HEIC support
register_heif_opener()

# === INPUT HEIC FILE ===
heic_file = "IMG_2820.heic"  # Replace with your file path

# === LOAD IMAGE ===
try:
    img = Image.open(heic_file)
    width, height = img.size
    print(f"📸 Resolution of '{heic_file}': {width} x {height} pixels")
except Exception as e:
    print(f"❌ Failed to read HEIC: {e}")


📸 Resolution of 'IMG_2820.heic': 3024 x 4032 pixels
